In [3]:
!pip install langchain langchain-groq langchain-community duckduckgo-search wikipedia -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [4]:
!pip install langchain==0.1.20 langchain-community langchain-groq duckduckgo-search wikipedia -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.1.53 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.0.2 which is incompatible.
opencv-pyt

In [5]:
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = "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"
from langchain.agents import AgentExecutor
from langchain.agents.react.agent import create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain.tools import Tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from datetime import datetime
import json, re, textwrap


In [6]:
# Tool 1 — Web Search
search_run = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    try:
        return f"[Web Search: {query}]\n{search_run.run(query)}"
    except Exception as e:
        return f"Search failed: {e}"

web_tool = Tool(
    name="web_search",
    func=web_search,
    description="Search the web for current info."
)

# Tool 2 — Wikipedia
wiki = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=3000)

def wiki_search(query: str) -> str:
    try:
        return f"[Wikipedia: {query}]\n{wiki.run(query)}"
    except Exception as e:
        return f"Wikipedia failed: {e}"

wiki_tool = Tool(
    name="wikipedia",
    func=wiki_search,
    description="Search Wikipedia for background knowledge."
)

tools = [web_tool, wiki_tool]
print("✅ Tools ready:", [t.name for t in tools])

✅ Tools ready: ['web_search', 'wikipedia']


In [7]:
def run_research(topic: str) -> dict:

    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.3,
        max_tokens=2048,
    )

    prompt = f"""
Write a detailed research report on: {topic}

Return ONLY valid JSON in this format:
{{
  "title": "",
  "introduction": "",
  "key_findings": ["", "", "", "", ""],
  "challenges": ["", "", ""],
  "future_scope": ["", "", ""],
  "conclusion": "",
  "references": ["", "", "", "", ""]
}}
"""

    try:
        response = llm.invoke(prompt)
        raw = response.content
    except Exception:
        return {}

    try:
        data = json.loads(raw)
    except:
        match = re.search(r"\{[\s\S]*\}", raw)
        data = json.loads(match.group()) if match else {}

    data.setdefault("title", topic)
    data.setdefault("introduction", "")
    data.setdefault("key_findings", [])
    data.setdefault("challenges", [])
    data.setdefault("future_scope", [])
    data.setdefault("conclusion", "")
    data.setdefault("references", [])

    return data


def print_report(r: dict):
    import textwrap

    print("\n" + "="*60)
    print(r["title"].upper())
    print("="*60)

    print("\nINTRODUCTION")
    print(textwrap.fill(r["introduction"], 70))

    print("\nKEY FINDINGS")
    for i, f in enumerate(r["key_findings"], 1):
        print(f"{i}. {f}")

    print("\nCHALLENGES")
    for i, c in enumerate(r["challenges"], 1):
        print(f"{i}. {c}")

    print("\nFUTURE SCOPE")
    for i, s in enumerate(r["future_scope"], 1):
        print(f"{i}. {s}")

    print("\nCONCLUSION")
    print(textwrap.fill(r["conclusion"], 70))

    print("\nREFERENCES")
    for i, ref in enumerate(r["references"], 1):
        print(f"{i}. {ref}")

    print("="*60 + "\n")


# 🔥 MAIN (silent execution)
topic = input("Enter topic: ").strip()

if topic:
    report = run_research(topic)
    print_report(report)
else:
    print("Invalid input")

Enter topic: Free AI tools in nursing for teaching the students

FREE AI TOOLS IN NURSING EDUCATION: A REVIEW OF RESOURCES FOR TEACHING STUDENTS

INTRODUCTION
The integration of Artificial Intelligence (AI) in nursing education
has the potential to revolutionize the way students learn and interact
with complex healthcare concepts. Free AI tools can provide nursing
educators with innovative methods to teach students, enhance
engagement, and improve learning outcomes. This report explores the
current landscape of free AI tools available for nursing education,
highlighting their applications, benefits, and limitations.

KEY FINDINGS
1. Chatbots like IBM Watson Assistant and Microsoft Bot Framework can be used to create interactive virtual teaching assistants that provide students with personalized learning experiences.
2. AI-powered simulation tools such as SimLab and Nursing Simulation Studio offer realistic and immersive learning environments for students to practice clinical skills.
3.